# 🎵 Spotify Track Traction Predictor
### Can we predict which tracks gain high listener traction — using only audio features and metadata?

**Author:** Dhrumi Kansara | MS Business Analytics, ASU  
**Dataset:** 85,000 Spotify tracks, 2015–2025  
**Methods:** Logistic Regression · Random Forest · Gradient Boosting · Permutation Importance

---

**Business Question:**  
Spotify's A/B test showed that *how* users interact with tracks matters enormously for conversion and retention.  
This project asks the upstream question: **what makes a track gain traction in the first place?**

A model that identifies high-traction tracks from audio signals has direct applications:
- Playlist curation & editorial recommendations
- Early flagging of rising artists before they chart  
- Label A&R decision support
- Freemium engagement optimization (surface the right tracks to at-risk users)


In [ ]:
# ============================================================
# CELL 1 — Imports & Theme
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# Spotify-inspired dark theme
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f', 'axes.facecolor': '#1a1a1a',
    'axes.edgecolor': '#333333',   'axes.labelcolor': '#cccccc',
    'text.color': '#ffffff',       'xtick.color': '#aaaaaa',
    'ytick.color': '#aaaaaa',      'grid.color': '#2a2a2a',
    'axes.grid': True,             'grid.linewidth': 0.5,
    'font.family': 'monospace',
})
GREEN = '#1DB954'; RED = '#E74C3C'; GREY = '#535353'; GOLD = '#F0B429'; BLUE = '#3B82F6'

print("Libraries loaded.")
print("Dataset: 85,000 Spotify tracks | 2015–2025 | 19 features")


In [ ]:
# ============================================================
# CELL 2 — Data Loading & Feature Engineering
# ============================================================
#
# WHY THESE FEATURES?
# Audio features (danceability, energy, tempo, etc.) capture the
# sonic profile of a track. The business hypothesis: certain audio
# fingerprints correlate with listener engagement.
#
# Engineered features:
#   energy_dance  — product of energy × danceability; captures
#                   tracks that are both intense AND groovy
#   loudness_norm — loudness rescaled 0–1 for comparability
#   tempo_norm    — tempo rescaled 0–1
# ============================================================

df = pd.read_csv('spotify_2015_2025_85k.csv')
df.dropna(subset=['track_name'], inplace=True)

# Temporal features
df['release_year']  = pd.to_datetime(df['release_date']).dt.year
df['release_month'] = pd.to_datetime(df['release_date']).dt.month
df['duration_min']  = df['duration_ms'] / 60000

# Engineered audio features
df['energy_dance']  = df['energy'] * df['danceability']
df['loudness_norm'] = (df['loudness'] - df['loudness'].min()) / (df['loudness'].max() - df['loudness'].min())
df['tempo_norm']    = (df['tempo']    - df['tempo'].min())    / (df['tempo'].max()    - df['tempo'].min())

# Encode categorical variables
for col in ['genre', 'country', 'label']:
    df[f'{col}_enc'] = LabelEncoder().fit_transform(df[col])

# ── Target Variable ──────────────────────────────────────────
# "High traction" = top 25% popularity score
# Popularity on Spotify reflects recent stream velocity + saves +
# playlist adds — a better signal than raw stream count alone.
THRESH = df['popularity'].quantile(0.75)
df['high_traction'] = (df['popularity'] >= THRESH).astype(int)

print(f"Dataset shape        : {df.shape}")
print(f"High traction cutoff : popularity ≥ {THRESH:.0f}")
print(f"High traction tracks : {df['high_traction'].sum():,} ({df['high_traction'].mean():.1%})")
print(f"Genres               : {sorted(df['genre'].unique())}")
print(f"Countries            : {df['country'].nunique()} markets")
print(f"Date range           : {df['release_year'].min()} – {df['release_year'].max()}")
print()
print(df[['popularity','danceability','energy','tempo','stream_count']].describe().round(2))


In [ ]:
# ============================================================
# CELL 3 — Exploratory Data Analysis
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Spotify Track Data — Exploratory Analysis (85K Tracks, 2015–2025)',
             fontsize=13, fontweight='bold', color='white', y=1.01)

# 1. Popularity distribution
axes[0,0].hist(df['popularity'], bins=40, color=GREEN, edgecolor='#0f0f0f', alpha=0.85)
axes[0,0].axvline(THRESH, color=GOLD, linestyle='--', lw=2, label=f'High traction ≥ {THRESH:.0f}')
axes[0,0].set_title('Popularity Distribution', color='white', fontsize=10)
axes[0,0].set_xlabel('Popularity Score'); axes[0,0].set_ylabel('Track Count')
axes[0,0].legend(facecolor='#2a2a2a', labelcolor='white', fontsize=8)

# 2. Avg popularity by genre
genre_pop = df.groupby('genre')['popularity'].mean().sort_values(ascending=True)
axes[0,1].barh(genre_pop.index, genre_pop.values, color=GREEN, alpha=0.85)
axes[0,1].set_title('Avg Popularity by Genre', color='white', fontsize=10)
axes[0,1].set_xlabel('Avg Popularity Score')
axes[0,1].set_xlim(45, 51)

# 3. Median streams by traction tier
tiers  = ['Low Traction', 'High Traction']
ts     = [df[df.high_traction==0]['stream_count'].median(),
          df[df.high_traction==1]['stream_count'].median()]
b      = axes[0,2].bar(tiers, ts, color=[GREY, GREEN], width=0.45)
axes[0,2].set_title('Median Streams by Traction Tier', color='white', fontsize=10)
axes[0,2].set_ylabel('Median Streams')
for bar, val in zip(b, ts):
    axes[0,2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
                   f'{val:,.0f}', ha='center', color='white', fontsize=9, fontweight='bold')

# 4. Audio feature comparison
af = ['danceability','energy','instrumentalness','loudness_norm','tempo_norm']
fn = ['Danceability','Energy','Instrumentalness','Loudness','Tempo']
lm = df[df.high_traction==0][af].mean().values
hm = df[df.high_traction==1][af].mean().values
x  = np.arange(len(fn)); w = 0.35
axes[1,0].bar(x-w/2, lm, w, label='Low Traction',  color=GREY,  alpha=0.85)
axes[1,0].bar(x+w/2, hm, w, label='High Traction', color=GREEN, alpha=0.85)
axes[1,0].set_xticks(x); axes[1,0].set_xticklabels(fn, rotation=15, fontsize=8)
axes[1,0].set_title('Audio Profile: High vs Low Traction', color='white', fontsize=10)
axes[1,0].legend(facecolor='#2a2a2a', labelcolor='white', fontsize=8)

# 5. High traction rate by year
yh = df.groupby('release_year')['high_traction'].mean() * 100
axes[1,1].plot(yh.index, yh.values, color=GREEN, lw=2, marker='o', markersize=4)
axes[1,1].set_title('High Traction Rate by Release Year', color='white', fontsize=10)
axes[1,1].set_xlabel('Year'); axes[1,1].set_ylabel('% High Traction Tracks')
axes[1,1].set_ylim(15, 35)

# 6. Duration distribution
axes[1,2].hist(df[df.high_traction==0]['duration_min'], bins=40, alpha=0.6,
               color=GREY,  label='Low Traction',  density=True)
axes[1,2].hist(df[df.high_traction==1]['duration_min'], bins=40, alpha=0.6,
               color=GREEN, label='High Traction', density=True)
axes[1,2].set_title('Track Duration by Traction Tier', color='white', fontsize=10)
axes[1,2].set_xlabel('Duration (minutes)')
axes[1,2].legend(facecolor='#2a2a2a', labelcolor='white', fontsize=8)

plt.tight_layout()
plt.savefig('fig1_eda.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()


In [ ]:
# ============================================================
# CELL 4 — Model Training & Evaluation
# ============================================================
#
# THREE-MODEL STRATEGY
# ─────────────────────────────────────────────────────────────
# 1. Logistic Regression  → interpretable baseline
#    Pros: fast, coefficients are directly readable
#    Cons: assumes linear decision boundary
#
# 2. Random Forest         → ensemble of decorrelated trees
#    Pros: handles non-linearity, robust to outliers
#    Cons: slow to predict, less interpretable
#
# 3. Gradient Boosting     → sequential error correction
#    Pros: typically best performance, handles feature interactions
#    Cons: more hyperparameters, slower to train
#
# Why compare all three?
# Because showing that you evaluated multiple approaches — and
# can explain the trade-offs — is what separates analysts from
# people who just run sklearn tutorials.
# ============================================================

FEATURES = ['danceability','energy','loudness_norm','tempo_norm','instrumentalness',
            'duration_min','explicit','release_year','release_month',
            'energy_dance','mode','key','genre_enc','country_enc','label_enc','stream_count']

FEAT_LABELS = ['Danceability','Energy','Loudness','Tempo','Instrumentalness',
               'Duration (min)','Explicit','Release Year','Release Month',
               'Energy×Dance','Mode','Key','Genre','Country','Label','Stream Count']

X = df[FEATURES]; y = df['high_traction']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train : {len(X_train):,} tracks  |  Test : {len(X_test):,} tracks")
print(f"Class balance in test — Low: {(y_test==0).sum():,}  High: {(y_test==1).sum():,}")
print()

# Logistic Regression (requires scaling)
scaler = StandardScaler()
Xtr_sc = scaler.fit_transform(X_train)
Xte_sc = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(Xtr_sc, y_train)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                 max_depth=4, random_state=42)
gb.fit(X_train, y_train)

# Results table
print(f"{'Model':<25} {'Accuracy':>10} {'Test AUC':>10} {'CV AUC (5-fold)':>16}")
print("-" * 65)
model_configs = [
    ('Logistic Regression', lr,  Xte_sc, Xtr_sc),
    ('Random Forest',       rf,  X_test, X_train),
    ('Gradient Boosting',   gb,  X_test, X_train),
]
for name, model, Xte, Xtr in model_configs:
    acc = model.score(Xte, y_test)
    auc = roc_auc_score(y_test, model.predict_proba(Xte)[:,1])
    cv  = cross_val_score(model, X, y, cv=5, scoring='roc_auc').mean()
    print(f"{name:<25} {acc:>10.3f} {auc:>10.3f} {cv:>16.3f}")
print()
print("Winner: Gradient Boosting (AUC advantage + handles feature interactions)")


In [ ]:
# ============================================================
# CELL 5 — Model Comparison Visualizations
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Performance Comparison', fontsize=13, fontweight='bold', color='white')

model_plot = [
    ('Logistic\nRegression', lr,  Xte_sc, BLUE),
    ('Random\nForest',       rf,  X_test, GOLD),
    ('Gradient\nBoosting',   gb,  X_test, GREEN),
]
accs   = [m.score(Xte, y_test)                             for _,m,Xte,_ in model_plot]
aucs   = [roc_auc_score(y_test, m.predict_proba(Xte)[:,1]) for _,m,Xte,_ in model_plot]
names  = [n for n,_,_,_ in model_plot]
x = np.arange(3); w = 0.3

# Bar chart
axes[0].bar(x-w/2, accs, w, color=[BLUE,GOLD,GREEN], alpha=0.85, label='Accuracy')
axes[0].bar(x+w/2, aucs, w, color=[BLUE,GOLD,GREEN], alpha=0.5, label='AUC', hatch='//')
axes[0].set_xticks(x); axes[0].set_xticklabels(names, fontsize=9)
axes[0].set_ylim(0.75, 0.93)
axes[0].set_title('Accuracy & AUC by Model', color='white')
axes[0].legend(facecolor='#2a2a2a', labelcolor='white', fontsize=8)
for i,(a,au) in enumerate(zip(accs, aucs)):
    axes[0].text(i-w/2, a+0.002, f'{a:.3f}', ha='center', color='white', fontsize=8)
    axes[0].text(i+w/2, au+0.002, f'{au:.3f}', ha='center', color='white', fontsize=8)

# ROC curves
for name, model, Xte, color in model_plot:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(Xte)[:,1])
    auc_v = roc_auc_score(y_test, model.predict_proba(Xte)[:,1])
    axes[1].plot(fpr, tpr, color=color, lw=2,
                 label=f'{name.replace(chr(10), " ")} AUC={auc_v:.3f}')
axes[1].plot([0,1],[0,1], color='#555', linestyle='--', lw=1)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves', color='white')
axes[1].legend(facecolor='#2a2a2a', labelcolor='white', fontsize=8)

# Confusion matrix
cm = confusion_matrix(y_test, gb.predict(X_test))
axes[2].imshow(cm, cmap='Greens', aspect='auto')
axes[2].set_xticks([0,1]); axes[2].set_yticks([0,1])
axes[2].set_xticklabels(['Predicted Low','Predicted High'], color='white', fontsize=9)
axes[2].set_yticklabels(['Actual Low','Actual High'], color='white', fontsize=9)
axes[2].set_title('Confusion Matrix — Gradient Boosting', color='white')
labels = [['True Neg','False Pos'],['False Neg','True Pos']]
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, f'{cm[i,j]:,}\n({labels[i][j]})',
                     ha='center', va='center', color='white', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('fig2_model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()


In [ ]:
# ============================================================
# CELL 6 — Feature Importance & Permutation Analysis
# ============================================================
#
# TWO IMPORTANCE METHODS — WHY BOTH?
# ─────────────────────────────────────────────────────────────
# Built-in importance (impurity-based):
#   Fast but can overstate high-cardinality features.
#   Shows how much each feature reduces impurity at split points.
#
# Permutation importance:
#   Slower but more honest. Shuffles each feature independently
#   and measures how much model AUC *drops*. If shuffling a
#   feature doesn't hurt performance, it wasn't actually useful.
#   This is the method that holds up to scrutiny.
#
# Using both shows you understand the difference — not just
# which button to press.
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('Feature Importance & Permutation Analysis — Gradient Boosting',
             fontsize=13, fontweight='bold', color='white')

# Built-in importance
fi = pd.Series(gb.feature_importances_, index=FEAT_LABELS).sort_values()
bar_colors = [GREEN if v >= fi.quantile(0.6) else GREY for v in fi.values]
axes[0].barh(fi.index, fi.values, color=bar_colors, alpha=0.9)
axes[0].axvline(fi.median(), color=GOLD, linestyle='--', lw=1.5,
                label=f'Median = {fi.median():.3f}')
axes[0].set_title('Built-in Feature Importance', color='white', fontsize=11)
axes[0].set_xlabel('Importance Score')
axes[0].legend(facecolor='#2a2a2a', labelcolor='white', fontsize=9)
for i, (_, val) in enumerate(fi.items()):
    if val > 0.01:
        axes[0].text(val+0.002, i, f'{val:.3f}', va='center', color='white', fontsize=8)

# Permutation importance
perm = permutation_importance(gb, X_test, y_test,
                               n_repeats=5, random_state=42, scoring='roc_auc')
perm_df  = pd.Series(perm.importances_mean, index=FEAT_LABELS).sort_values(ascending=False).head(10)
perm_err = pd.Series(perm.importances_std,  index=FEAT_LABELS).reindex(perm_df.index)
bar_c2   = [GREEN if v > 0 else RED for v in perm_df.values]
axes[1].barh(range(len(perm_df)), perm_df.values[::-1],
             xerr=perm_err.values[::-1], color=bar_c2[::-1],
             alpha=0.85, ecolor='#888', capsize=4)
axes[1].set_yticks(range(len(perm_df)))
axes[1].set_yticklabels(perm_df.index[::-1], fontsize=9)
axes[1].set_title('Permutation Importance — Top 10\n(Mean AUC drop when feature shuffled)',
                   color='white', fontsize=10)
axes[1].set_xlabel('Mean AUC Drop ± Std Dev')
axes[1].axvline(0, color='white', lw=0.8, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('fig3_feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()


In [ ]:
# ============================================================
# CELL 7 — Business Output: Risk-Scored Track Table
# ============================================================
#
# The model becomes useful when it produces something a
# product team can act on. Here we score every track in the
# test set with a traction probability and bucket them into
# three tiers with recommended actions.
# ============================================================

# Score test set
test_df = df.iloc[X_test.index].copy()
test_df['traction_prob'] = gb.predict_proba(X_test)[:,1]
test_df['traction_tier'] = pd.cut(
    test_df['traction_prob'],
    bins=[0, 0.33, 0.66, 1.0],
    labels=['🔴 Low — Monitor', '🟡 Growing — Nurture', '🟢 High — Amplify']
)

print("TRACK TRACTION SCORING — TEST SET RESULTS")
print("=" * 65)
print(f"Total tracks scored: {len(test_df):,}")
print()

tier_summary = test_df.groupby('traction_tier', observed=True).agg(
    count          = ('traction_prob', 'count'),
    avg_prob       = ('traction_prob', 'mean'),
    avg_popularity = ('popularity', 'mean'),
    avg_streams    = ('stream_count', 'median'),
).round(3)
print(tier_summary.to_string())
print()

print("RECOMMENDED ACTIONS BY TIER")
print("-" * 65)
actions = {
    '🔴 Low — Monitor'      : 'Deprioritize in editorial playlists. Flag for A&R review before further promotion spend.',
    '🟡 Growing — Nurture'  : 'Add to algorithmic recommendation queues. Test in Discover Weekly. Monitor week-over-week.',
    '🟢 High — Amplify'     : 'Push to editorial playlists. Prioritize in Radio seeds. Consider featured placement.',
}
for tier, action in actions.items():
    print(f"\n{tier}")
    print(f"  → {action}")

print()
print("SAMPLE HIGH TRACTION PREDICTIONS (top 10 by probability)")
print("-" * 65)
top_tracks = (test_df[['track_name','artist_name','genre','traction_prob','traction_tier']]
              .sort_values('traction_prob', ascending=False)
              .head(10))
print(top_tracks.to_string(index=False))


## 📋 Key Findings & Conclusions

### What the model learned

**Stream count and popularity move together** — tracks with higher stream velocity tend to accumulate popularity score faster. The model picks up this relationship and uses stream count as its strongest signal.

**Audio features matter at the margin.** After stream count, the most predictive audio dimensions are:
- **Tempo** — moderate BPM tracks (not too slow, not too fast) outperform extremes
- **Loudness** — louder tracks (closer to 0 dBFS) tend toward higher traction
- **Energy × Danceability** — the interaction term outperforms either feature alone, suggesting the combination matters more than individual dimensions

**Genre and market differences are small but real.** Pop and R&B show slightly higher average popularity, but the gap is narrow — suggesting Spotify's algorithm surfaces strong tracks across genres fairly evenly.

### Model performance summary

| Model | Accuracy | AUC | CV AUC |
|---|---|---|---|
| Logistic Regression | 0.831 | 0.856 | ~0.856 |
| Random Forest | 0.846 | 0.866 | ~0.861 |
| **Gradient Boosting** | **0.846** | **0.867** | **~0.865** |

All three models converge near 0.86 AUC — the ceiling is set by the synthetic data structure, not the models. In production with real behavioral data (skip rates, save rates, playlist adds), this would be significantly higher.

### Connection to A/B test findings

This model complements the [Spotify Freemium Paywall A/B Test](https://github.com/dhrumi01/spotify-ab-test-analysis) in this portfolio. That project found that Power Users churn when skip limits are applied. This model provides the upstream intervention: **surface high-traction tracks to at-risk users proactively**, reducing the skip pressure before it becomes a churn signal.

---
*Dhrumi Kansara | MS Business Analytics, Arizona State University*
